# 🚌 버스 혼잡도 예측 AI 연구소
**7/8차시 | 바이브 코딩 입문 시리즈**

---

| 단계 | 셀 | 할 일 | 수정? |
|:---:|---|---|:---:|
| 준비 | 1–1 | 라이브러리 설치 | ❌ |
| 준비 | 1–2 | 라이브러리 import | ❌ |
| 준비 | 2 | CSV 업로드 | ❌ |
| 준비 | 3 | 데이터 확인 | ❌ |
| 학습 | 4–1 | 모델 학습 & 정확도 확인 | ❌ |
| **테스트** | **4–2** | **예측 직접 해보기** | **✅ 수정!** |
| 저장 | 5–1 | ONNX 저장 | ❌ |
| 저장 | 5–2 | stop_metadata.json 저장 | ❌ |
| 저장 | 5–3 | model_info.json 저장 | ❌ |
| 저장 | 5–4 | 파일 다운로드 | ❌ |
| 검증 | 6 | ONNX 무결성 검사 | ❌ |

> 📂 **사용 데이터**: `bus_labeled.csv` (5/6차시에서 만든 파일)  
> ✏️ **수정이 필요한 셀은 4–2 하나뿐입니다. 나머지는 순서대로 실행하세요.**

---
## 🔧 준비 단계
### 셀 1–1 | 라이브러리 설치
> 모델 학습에 필요한 라이브러리를 설치하는 코드입니다.

In [ ]:
!pip install skl2onnx onnxruntime -q
print('✅ 설치 완료!')

✅ 설치 완료!


### 셀 1–2 | 라이브러리 Import

In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType
import onnxruntime as rt

print('✅ 모든 라이브러리 로드 완료!')

✅ 모든 라이브러리 로드 완료!


---
### 셀 2 | CSV 업로드
> 왼쪽 사이드바의 📁 폴더 아이콘을 눌러 `bus_labeled.csv` 파일을 드래그하여 업로드한 뒤 실행합니다.

In [ ]:
df = pd.read_csv('bus_labeled.csv', encoding='utf-8-sig')

print('✅ 파일 읽기 완료!')
print(f'전체 데이터 수 : {len(df):,} 개')
display(df.head())

✅ 파일 읽기 완료!
전체 데이터 수 : 165,480 개


,hour,stop_id,stop_name,total,label
0,0,23849,한남대교(가상)(00015),0,0
1,0,23849,한남대교(가상)(00029),3,0
2,0,23849,한남대교(가상)(00039),0,0
3,0,23850,한남대교(가상)(00011),0,0
4,0,23850,한남대교(가상)(00027),0,0


---
### 셀 3 | 데이터 확인
> 레이블 분포를 확인합니다. 5/6차시에서 기록한 비율과 같은지 확인합니다.

In [ ]:
print('=== 레이블 분포 ===')
counts = df['label'].value_counts().sort_index()
names = {0: '🟢 여유(0)', 1: '🟡 보통(1)', 2: '🔴 혼잡(2)'}
for label, count in counts.items():
    pct = count / len(df) * 100
    bar = '█' * int(pct / 2)
    print(f'{names[label]:12s} : {count:6,}개  ({pct:5.1f}%)  {bar}')

print(f'\ntotal 범위 : {df["total"].min():.0f}명 ~ {df["total"].max():.0f}명')
print(f'total 평균 : {df["total"].mean():.0f}명')

=== 레이블 분포 ===
🟢 여유(0)      : 48,683개  ( 29.4%)  ██████████████
🟡 보통(1)      :  8,825개  (  5.3%)  ██
🔴 혼잡(2)      : 107,972개  ( 65.2%)  ████████████████████████████████

total 범위 : 0명 ~ 8060명
total 평균 : 316명


---
## 🤖 학습 단계
### 셀 4 | 모델 학습 & 정확도 확인
> `hour`와 `stop_id`를 입력값으로, `label`을 정답으로 하여 모델을 학습시킵니다.  
> 정확도를 학습지에 기록하세요!

In [ ]:
X = df[['hour', 'stop_id']].values.astype(np.float32)
y = df['label'].values

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'학습용 데이터 수  : {len(X_tr):,} 개  (전체의 80%)')
print(f'테스트용 데이터 수 : {len(X_te):,} 개  (전체의 20%)')
print()
print('⏳ 모델 학습 중입니다. 잠시 기다려주세요...')

model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
model.fit(X_tr, y_tr)
acc = accuracy_score(y_te, model.predict(X_te))

print('━' * 40)
print(f'  🏆 정확도: {acc*100:.1f}%')
print('━' * 40)

if   acc >= 0.90: print('  🌟 탁월! 실제 서비스 수준이에요!')
elif acc >= 0.75: print('  ✅ 우수! 예측에 충분해요.')
elif acc >= 0.60: print('  🔧 보통.')
else:             print('  💪 5/6차시 레이블 기준을 다시 확인해보세요.')

학습용 데이터 수  : 132,384 개  (전체의 80%)
테스트용 데이터 수 : 33,096 개  (전체의 20%)

⏳ 모델 학습 중입니다. 잠시 기다려주세요...
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  🏆 정확도: 81.5%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅ 우수! 예측에 충분해요.


---
## 🔍 예측 테스트
### 셀 4–2 | 직접 예측해보기
> 시간대와 정류장 번호를 바꿔가며 AI의 예측 결과를 확인합니다.  
> **✏️ 이 두 줄만 수정하세요.**

In [ ]:
# ✏️ 여기만 수정하세요
테스트_시간대 = 2      # 0~23 사이의 숫자
테스트_정류장 = 23281  # bus_labeled.csv에 있는 stop_id 데이터 (23281은 역삼역7번출구.GS타워의 stop_id)


# ---- 아래는 수정하지 않습니다 ----
result = model.predict([[float(테스트_시간대), float(테스트_정류장)]])[0]
names  = {0: '🟢 여유', 1: '🟡 보통', 2: '🔴 혼잡'}

print(f'입력값  → 시간대 : {테스트_시간대}시 / 정류장 번호 : {테스트_정류장}')
print(f'예측 결과 → {names[result]}')

입력값  → 시간대 : 2시 / 정류장 번호 : 23281
예측 결과 → 🟢 여유


---
## 💾 저장 단계 — 보물 3대장
### 셀 5–1 | bus_model.onnx 저장
> 학습된 AI 모델을 브라우저에서 실행 가능한 ONNX 포맷으로 변환·저장합니다.

In [ ]:
onnx_model = convert_sklearn(
    model,
    initial_types=[('float_input', FloatTensorType([None, 2]))],
    target_opset=11
)
with open('bus_model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

print('✅ bus_model.onnx 저장 완료')

✅ bus_model.onnx 저장 완료


### 셀 5–2 | stop_metadata.json 저장
> 정류장 번호 목록과 시간대 정보를 저장합니다. 웹앱에서 드롭다운을 채우는 데 사용합니다.

In [ ]:
stop_ids = sorted(df['stop_id'].unique())

meta = []
for sid in stop_ids:
    sub = df[df['stop_id'] == sid]
    peak_hour = int(sub.groupby('hour')['total'].mean().idxmax())
    meta.append({
        'stop_id':   int(sid),
        'stop_name': str(sub['stop_name'].iloc[0]),
        'peak_hour': peak_hour,
        'avg_total': round(float(sub['total'].mean()), 1),
    })

with open('stop_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

print(f'✅ stop_metadata.json 저장 완료 ({len(meta):,}개 정류장)')

✅ stop_metadata.json 저장 완료 (503개 정류장)


### 셀 5–3 | model_info.json 저장
> 이 모델이 사용한 피처 목록을 저장합니다. 웹앱이 입력값 순서를 읽는 데 사용합니다.

In [ ]:
with open('model_info.json', 'w', encoding='utf-8') as f:
    json.dump({
        'features': ['hour', 'stop_id'],
        'accuracy': round(float(acc), 4),
    }, f, ensure_ascii=False, indent=2)

print('✅ model_info.json 저장 완료')

✅ model_info.json 저장 완료


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 셀 5–4 | 보물 3대장 다운로드
> 세 파일을 PC로 다운로드합니다. `index.html`과 **같은 폴더**에 넣어야 웹앱이 동작합니다.

In [ ]:
from google.colab import files

files.download('bus_model.onnx')
files.download('stop_metadata.json')
files.download('model_info.json')

print('📥 다운로드 완료!')
print('   🧠 bus_model.onnx')
print('   🗺️  stop_metadata.json')
print('   📋 model_info.json')
print()
print('➡️  index.html과 같은 폴더에 보관하세요!')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 다운로드 완료!
   🧠 bus_model.onnx
   🗺️  stop_metadata.json
   📋 model_info.json

➡️  index.html과 같은 폴더에 보관하세요!


---
## 🔎 검증 단계
### 셀 6 | ONNX 무결성 자동 검증
> Python 모델과 ONNX 모델의 예측값이 일치하는지 확인합니다.  
> **✅ 검증 성공** 메시지가 나와야 웹앱에서 정상 동작합니다.

In [ ]:
try:
    sess       = rt.InferenceSession('bus_model.onnx')
    input_name = sess.get_inputs()[0].name
    onnx_pred  = sess.run(None, {input_name: X_te})[0]
    py_pred    = model.predict(X_te)
    match_rate = (onnx_pred == py_pred).mean()

    if match_rate >= 0.999:
        print('✅ [검증 성공] Python 모델과 ONNX 모델의 예측이 일치합니다!')
        print(f'   정확도 : {acc*100:.1f}%')
        print()
        print('🚀 보물 3대장을 가지고 바이브 코딩으로 넘어가세요!')
    else:
        print(f'⚠️  [경고] 일치율 {match_rate*100:.1f}% — 셀 4를 다시 실행해보세요.')
except Exception as e:
    print(f'❌ [검증 실패] {e}')

✅ [검증 성공] Python 모델과 ONNX 모델의 예측이 일치합니다!
   정확도 : 81.5%

🚀 보물 3대장을 가지고 바이브 코딩으로 넘어가세요!
